In [ ]:
from __future__ import annotations
import pandas as pd
import numpy as np
import scanpy as sc
import anndata as ad
import warnings
warnings.filterwarnings("ignore")
import hdf5plugin #in case needed for h5ad reading (if previously saved with compression)
import sys
import os

# input_path1 = str(sys.argv[1])
# input_path2 = str(sys.argv[2])

input_path1 = "/home/sofia/Projects/etmr/defaria/scRNA/jessa/data/processed/so_etmr1_anotado_filt.h5ad"
input_path2 = "/home/sofia/Projects/etmr/defaria/scRNA/paper_data/data/processed/paper_data_filt.h5ad"


print("Reading" + input_path1 + "...")
etmr1 = ad.io.read_h5ad(input_path1)
print(etmr1)

print("Reading" + input_path2 + "...")
etmr2 = ad.io.read_h5ad(input_path2)
print(etmr2)


print("Computing" + input_path1 + " PCA...")
sc.pp.pca(etmr1)


Reading/home/sofia/Projects/etmr/defaria/scRNA/jessa/data/processed/so_etmr1_anotado_filt.h5ad...
AnnData object with n_obs × n_vars = 6363 × 4220
    obs: 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'percent.ribo', 'nf', 'RNA_snn_res.0.8', 'seurat_clusters', 'cell_types_etmr1', 'cell_types_2_etmr1', 'S.Score', 'G2M.Score', 'Phase', 'cluster_cell_type', 'cluster_cell_type_phase', 'cluster_cell_type_condition', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt'
    var: 'names', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'hvg', 'log1p'
    obsm: 'X_pca', 'X_umap.unintegrated'
    layers: 'counts'
Reading/home/sofia/Projects/etmr/defaria/scRNA/paper_data/data/processed/paper_data_filt.h5ad...
AnnData object with n_obs × n_vars = 998 × 6653
    obs: 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt'
    var: 'gene_ids', 'fe

Now let's try different integration methods

In [ ]:
print("Computing" + input_path1 + "neighbors and UMAP...")
sc.pp.neighbors(etmr1, n_neighbors=10, n_pcs=50)
sc.tl.umap(etmr1)

print("Computing" + input_path1 + "Leiden clustering...")
sc.tl.leiden(etmr1, resolution=1)


print("Ingesting" + input_path2 + "...")
etmr2_int = etmr2.copy()

sc.tl.ingest(etmr2_int, etmr1, obs=['leiden', 'cell_types_etmr1', 'cell_types_2_etmr1'], embedding_method = "umap")

print("Merging datasets...")
all_data = ad.concat([etmr1, etmr2_int], label="condition", keys=["jessa", "defaria"])
all_data.obs["leiden"] = (
    all_data.obs["leiden"].astype("category").cat.reorder_categories(etmr1.obs["leiden"].cat.categories)
)



# fix category colors
for key in etmr1.uns:
    if key.endswith("_colors"):
        all_data.uns[key] = etmr1.uns[key]

In [2]:
all_data

AnnData object with n_obs × n_vars = 7361 × 1670
    obs: 'cell_types_etmr1', 'cell_types_2_etmr1', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'leiden', 'condition'
    obsm: 'X_umap'
    layers: 'counts'

In [9]:
etmr2_int.obs_names.intersection(etmr1.obs_names)

Index(['CATGCCTGTATGAAAC-1'], dtype='object')

In [16]:
etmr1.obs[etmr1.obs_names == 'CATGCCTGTATGAAAC-1']

,nCount_RNA,nFeature_RNA,percent.mt,percent.ribo,nf,RNA_snn_res.0.8,seurat_clusters,cell_types_etmr1,cell_types_2_etmr1,S.Score,G2M.Score,Phase,cluster_cell_type,cluster_cell_type_phase,cluster_cell_type_condition,n_genes_by_counts,total_counts,total_counts_mt,pct_counts_mt,leiden
CATGCCTGTATGAAAC-1,4202.0,2245,1.880057,8.995716,0.666397,2,2,2.Neuron,Neuron,-0.116891,-0.129749,G1,2.Neuron,2.Neuron,2.Neuron_tumor,2245,4202.0,79.0,1.880057,16


In [19]:
etmr2.obs[etmr2.obs_names == 'CATGCCTGTATGAAAC-1']

,n_genes_by_counts,total_counts,total_counts_mt,pct_counts_mt
CATGCCTGTATGAAAC-1,4367,14629.0,612.0,4.183471


In [6]:
etmr2.obs

,n_genes_by_counts,total_counts,total_counts_mt,pct_counts_mt
AAACCCAGTAGACGTG-1,3104,9248.0,218.0,2.357266
AAACGCTGTTCCGCGA-1,1206,2751.0,92.0,3.344239
AAACGCTTCGTTCTCG-1,406,532.0,10.0,1.879699
AAAGAACGTATTAAGG-1,4813,21843.0,881.0,4.033329
AAAGGGCTCATAGACC-1,1729,4039.0,123.0,3.045308
...,...,...,...,...
TTTCCTCGTAACTAAG-1,3269,7386.0,317.0,4.291903
TTTGATCGTTCGGTAT-1,3032,6506.0,235.0,3.612051
TTTGGAGAGACAGCTG-1,2976,8643.0,250.0,2.892514
TTTGGTTAGACGCATG-1,4911,19084.0,360.0,1.886397
